# Word vectors trained on SE articles and SE Glossary articles. Application for the identification of Eurostat datasets

### Installation instructions
*    For the setup of the Virtuoso ODBC data source please see section 1a in https://github.com/eurostat/NLP4Stat/tree/testing/Software%20Environment
*    Download the notebook as "raw" file and save it with extension .ipynb (cut the .txt extension which is added)
*    Install the necessary libraries from your jupyter command prompt. These, together with the versions used, are:
    *    pyodbc==4.0.32
    *    scipy==1.7.1
    *    pandas==1.3.5
    *    numpy==1.20.3
    *    nltk==3.6.5
    *    ipywidgets==7.6.5
    *    gensim==4.1.2
    *    eurostat==0.2.3

*   Launch the notebook and put your own credentials in the pyodbc.connect() call in the chunk with title "Glossary articles"

In [ ]:
import gensim
import pandas as pd
import numpy as np

import re

import pyodbc

## The following processing of SE and SE Glossary articles is for the first run only, to save the model. After this, it suffices to load the model



### Glossary articles  

* Definitions from dat_glossary.
* Titles from dat_link_info (with resource_information_id=1, i.e. Eurostat, see ESTAT.V1.mod_resource_information).
* Match above on id.


In [ ]:
c = pyodbc.connect('DSN=Virtuoso All;DBA=ESTAT;UID=xxxxx;PWD=xxxxx')
cursor = c.cursor()

SQLCommand = """SELECT T1.id, T1.definition, T2.title 
                FROM ESTAT.V1.dat_glossary as T1 
                INNER JOIN ESTAT.V1.dat_link_info as T2  
                  ON T1.id=T2.id 
                WHERE T2.resource_information_id=1 """

GL_df = pd.read_sql(SQLCommand,c)
GL_df = GL_df[['id', 'title', 'definition']]

GL_df


### Delete records with empty definitions and carry out data cleansing


In [ ]:

GL_df = GL_df.dropna(axis=0,how='any')
print(GL_df.isnull().sum())
GL_df.reset_index(drop=True, inplace=True)

#import unicodedata as ud

def clean(x, quotes=True):
    if pd.isnull(x): return x  
    x = x.strip()
    
    ## make letter-question mark-letter -> letter-quote-space-letter !!! but NOT in the lists of URLs!!!
    if quotes:
        x = re.sub(r'([A-Za-z])\?([A-Za-z])','\\1\' \\2',x) 
    
    ## make letter-question mark-space lower case letter letter-quote-space letter
    x = re.sub(r'([A-Za-z])\? ([a-z])','\\1\' \\2',x) 

    ## delete ,000 commas in numbers    
    x = re.sub(r'\b(\d+),(\d+)\b','\\1\\2',x) ## CORRECTED
    
    ## delete  000 spaces in numbers
    x = re.sub(r'\b(\d+) (\d+)\b','\\1\\2',x) ## CORRECTED
    
    ## remove more than one spaces
    x = re.sub(r' +', ' ',x)
    
    ## remove start and end spaces
    x = re.sub(r'^ +| +$', '',x,flags=re.MULTILINE) 
    
    ## space-comma -> comma
    x = re.sub(r' \,',',',x)
    
    ## space-dot -> dot
    x = re.sub(r' \.','.',x)
    
    x = re.sub(r'?.{2}',"'",x) ### !!! NEW: single quotes are read as: ?XX
    
    #x = x.encode('latin1').decode('utf-8') ## ?\x80\x99
    #x = ud.normalize('NFKD',x).encode('ascii', 'ignore').decode()
    
    return x


GL_df['title'] = GL_df['title'].apply(clean)
GL_df['title'] = GL_df['title'].apply(lambda x: re.sub(r'\?','-',x)) ## also replace question marks by dashes
GL_df['definition'] = GL_df['definition'].apply(clean)

GL_df.head(5)


### Delete "special" records

* i.e. redirections.



In [ ]:
## Drop The records with definitions "The revision ..." and "Redirect to ..." 

idx = GL_df[GL_df['definition'].str.startswith('The revision #')].index
print(idx)
GL_df.drop(idx , inplace=True)
idx = GL_df[GL_df['definition'].str.startswith('Redirect to')].index
print(idx)
GL_df.drop(idx , inplace=True)
GL_df.reset_index(drop=True, inplace=True)
GL_df

### Create column "raw content" with the titles and the definitions

In [ ]:
GL_df['raw content'] = GL_df['title'] +'. '+GL_df['definition']
#GL_df['source'] = 'GL'
GL_df
GL_df.to_excel('Word_vectors_GL_df.xlsx')

### Statistics Explained articles

* IDs, titles from dat_link_info, with resource_information_id=1, i.e. Eurostat (see ESTAT.V1.mod_resource_information) and matching IDs from dat_article.
* Carry out data cleansing on titles.

In [ ]:
SQLCommand = """SELECT id, title 
                FROM ESTAT.V1.dat_link_info 
                WHERE resource_information_id=1 AND id IN (SELECT id FROM ESTAT.V1.dat_article) """

SE_df = pd.read_sql(SQLCommand,c)

SE_df['title'] = SE_df['title'].apply(clean)
SE_df.head(5)

### Add paragraphs titles and contents

* From dat_article_paragraph with abstract=0 (i.e. "no").
* Match article_id from dat_article_paragraph with id from dat_article.
* Carry out data cleansing on titles and paragraph contents.

In [ ]:
SQLCommand = """SELECT article_id, title, content 
                FROM ESTAT.V1.dat_article_paragraph
                WHERE abstract=0 AND article_id IN (SELECT id FROM ESTAT.V1.dat_article) """

add_content = pd.read_sql(SQLCommand,c)
add_content['title'] = add_content['title'].apply(clean)
add_content['content'] = add_content['content'].apply(clean)
add_content

### Aggregate above paragraph titles and contents  from SE articles paragraphs by article id

* Create a column _raw content_ which gathers all paragraph titles and contents in one text per article.

In [ ]:
add_content_grouped = add_content.groupby(['article_id'])[['title','content']].aggregate(lambda x: list(x))
add_content_grouped.reset_index(drop=False, inplace=True)
for i in range(len(add_content_grouped)):
    add_content_grouped.loc[i,'raw content'] = ''
    for (a,b) in zip(add_content_grouped.loc[i,'title'],add_content_grouped.loc[i,'content']):
        add_content_grouped.loc[i,'raw content'] += a + '. ' + b
add_content_grouped = add_content_grouped[['article_id','raw content']]    

add_content_grouped

### Merge raw content of SE articles with main file

* Add the title to column "raw content".

In [ ]:
SE_df = pd.merge(SE_df,add_content_grouped,left_on='id',right_on='article_id',how='inner')
SE_df.drop(['article_id'],axis=1,inplace=True)

SE_df['raw content'] = SE_df['title'] +'. ' + SE_df['raw content']
#SE_df['source'] = 'SE'

SE_df.head(5)
SE_df.to_excel('Word_vectors_SE_df.xlsx')

### Concatenate the two dataframes 

In [ ]:
all_df = pd.concat([GL_df[['id','title','raw content']],SE_df[['id','title','raw content']]],ignore_index=True)
all_df

### Create 'sentences' list from the 'raw content' column after tokenization and lemmatization

* Keep terms with only alphanumeric characters and drop stop words.
* Drop numbers.
* Keep lemmas from original terms with at least 5 characters.

In [ ]:
import nltk

from nltk.corpus import stopwords

w_tokenizer = nltk.tokenize.WordPunctTokenizer()
lemmatizer = nltk.stem.WordNetLemmatizer()

stop = stopwords.words('english')

def lemmatize_text(text): ## only alphanumeric characters and drop stop-words
    return [lemmatizer.lemmatize(w).lower() for w in w_tokenizer.tokenize(text) if w.isalnum() and not w.lower() in stop
           and not re.match(r'^[0-9]+$',w) and len(w) >=5]

all_df['raw content']=all_df['raw content'].apply(lambda x: lemmatize_text(x))
##all_df         
sentences = all_df['raw content'].to_list()
##sentences

del SE_df, GL_df #, all_df

## Train word vectors on the 'sentences' list and save the trained model in binary file SE_GL_wordvectors.bin

## Processing can then start from the loading of the saved model

### Save the model also in plain text format: 'SE_GL_wordvectors.txt'.
* This has the number of vectors and the dimensionality in the first row.
* Then, for each word in the model, the coordinates of the corresponding vector.


In [ ]:
from gensim.models import Word2Vec
from gensim.models import KeyedVectors

from gensim.parsing.preprocessing import STOPWORDS
from gensim.parsing.porter import PorterStemmer

p = PorterStemmer()
all_stopwords_gensim = STOPWORDS

print('Type of corpus: ', type(sentences))
print('Length of corpus: ', len(sentences))

model = Word2Vec(sentences=sentences,vector_size=300,workers=4,epochs=10) # number of dimensions: 300, 4 threads,
## epochs = 2 x default

word_vectors = model.wv
word_vectors.save("SE_GL_wordvectors.bin") ## store only vectors - not going to update the model

## save also as plain text
word_vectors.save_word2vec_format('SE_GL_wordvectors.txt', binary=False)

del sentences

model = KeyedVectors.load("SE_GL_wordvectors.bin", mmap='r') 
print('Number of vectors: ',len(model))

## examples of use
for i in range(10):
    w = model.index_to_key[i]
    print(w,lemmatizer.lemmatize(w),p.stem(w),end='')
    if w in all_stopwords_gensim:
        print(' ***')
    else:
        print()
        similarities = model.most_similar(w,topn=10)
        for w2, score in similarities:
            print('    ',w2,score)

## Read the Eurostat's database table of contents

* From the parsed table_of_contents.xml file, stored in the database.
* Keep only records - leaves, corresponding to datasets.
* Column 'Normalized file description': from column 'File description', keeping terms with only alphanumeric characters (but not numbers), dropping stop words, lemmatizing and keeping lemmas from original terms with at least 5 characters.
* Column 'Normalized full path': as above from column 'Names'.

In [ ]:
import ast

SQLCommand = """SELECT id, number, codes, names, file_descr, file_code, level 
                FROM ESTAT.V1.dat_all_datasets """

crumbs_df = pd.read_sql(SQLCommand,c)

crumbs_df['codes'] = crumbs_df['codes'].apply(lambda x: ast.literal_eval(x))
crumbs_df['names'] = crumbs_df['names'].apply(lambda x: ast.literal_eval(x))
crumbs_df.rename(columns={'number':'Number','codes':'Codes','names':'Names','file_descr':'File description','level':'Level','file_code':'File code'},inplace=True)
crumbs_df = crumbs_df[['Number','Codes','Names','File description','File code','Level']]

## Keep only records - leaves, corresponding to datasets.
idx = crumbs_df[crumbs_df['File code']==''].index
crumbs_df.drop(index=idx, inplace=True) 
crumbs_df.reset_index(drop=True,inplace=True)

def lemmatize_list(x):
    str = ' '.join(x)
    return ' '.join(lemmatize_text(str))

crumbs_df['Normalized file description'] = crumbs_df['File description'].apply(lambda x: x.lower())
crumbs_df['Normalized file description'] = crumbs_df['Normalized file description'].apply(lambda x: ' '.join(lemmatize_text(x)))

crumbs_df['Normalized full path'] = crumbs_df['Names'].apply(lambda x: [y.lower() for y in x[1:]]) ## exclude first part
crumbs_df['Normalized full path'] = crumbs_df['Normalized full path'].apply(lemmatize_list)

crumbs_df


### The function scanning the datasets

* Create a 0-vector 'scores' of dimension # datasets
* Function scan_datasets():
    *    Average the vectors of words found in the input sentence using function 'avg_feature_vector()'.
    *    For each entry in the datasets column 'Normalized file description' (option=1) or in the datasets column 'Normalized full path' (option=2), similarly average the vectors of words found.
    *    Calculate the similarity of the input sentence with each dataset description, by calculating the cosine similarity of the two averaged vectors (function sent_similarity()).  
    *    Assign the similarities as scores in vector 'scores'. 


In [ ]:
index2_to_key_set = set(model.index_to_key)
num_features=300

from scipy import spatial

#import warnings
#warnings.filterwarnings("error")

def avg_feature_vector(sentence, model, num_features, index2_to_key_set):
    words = sentence.split()
    feature_vec = np.zeros((num_features, ), dtype='float32')
    n_words = 0
    for word in words:
        if word in index2_to_key_set:
            n_words += 1
            feature_vec = np.add(feature_vec, model[word])
    if (n_words > 0):
        feature_vec = np.divide(feature_vec, n_words)
    return feature_vec

def sent_similarity(sent1,sent2):

    s1_afv = avg_feature_vector(sent1, model=model, num_features=num_features, index2_to_key_set=index2_to_key_set)
    s2_afv = avg_feature_vector(sent2, model=model, num_features=num_features, index2_to_key_set=index2_to_key_set)
    if all(s1_afv==0.) or all(s2_afv==0.):
        sim=0.
    else:
 #       try:
        sim = 1 - spatial.distance.cosine(s1_afv, s2_afv)
#       except RuntimeWarning:
#            print(s1_afv)
#            print(s2_afv)
    return sim

def scan_datasets(lemmas, option):
    scores = np.zeros(len(crumbs_df))
    sentence = ' '.join(lemmas)
    if option==1:
        col = 'Normalized file description'
    else:
        col = 'Normalized full path'    
    for j in range(len(crumbs_df)):
        file_desc= crumbs_df.loc[j,col]
        scores[j] = sent_similarity(sentence,file_desc)
    return(scores)   



In [ ]:
import ipywidgets as widgets
from ipywidgets import Layout
layout = widgets.Layout(width='600px', height='60px')

In [ ]:
%%javascript
IPython.OutputArea.prototype._should_scroll = function(lines) {
    return false;
}

In [ ]:
from IPython.display import HTML, display,clear_output

### The function producing the HTML output to display
*    Sentences contains the text entered in the Textarea widget.
*    option = 1 to use the simple descriptions of the datasets or 2 for the full paths.
*    howmany: display this much number of datasets
*    show_meta: True to display metadata for a selected dataset
*    show_meta_ind: zero-based index in the results for the display of metadata

In [ ]:
import eurostat
#import qgrid

codes = {'hazard': 16, 'time': 1169, 'nace_r1': 888, 'geo': 4017, 
    'age691': None, 'rbd': 412, 'time1': None, 'nace_r2': 1298, 
    'worktime': 31, 'metroreg': 422, 'geo91': None, 'injury': 50, 'physact': 22, 'ceparema': 37, 
    'modinj': 11, 'train': 12, 'wrkenv': 15, 'frequenc': 61, 'tra_cov': 32, 'stk_flow': 190, 'par_mar': 302, 
    'currency': 193, 'cities': 13507, 'indic_env': 81, 'emp_cont': 17, 'loadstat': 3, 'size_emp': 31, 'airp_pr': 25160, 
    'unit': 668, 'nst07': 104, 'wat_proc': 56, 'comspec': 3, 'sex': 7, 'time91': None, 'age': 651, 
    'workproc': 9, 'age91': None, 'citizen': 4017, 'port_iww': 1032, 'source': 14, 'c_cabot': 4017, 
    'wrkstat': 5, 'c_birth': 4017, 'rep_airp': 1116, 'deviatn': 11, 'matagent': 23, 'diagnose': 28, 
    'rep_mar': 1842, 'partner': 4017, 'c_regis': 4017, 'geo1': None, 'seabasin': 12}
        

def relevant_datasets(Sentences,option,howmany,show_meta,show_meta_ind):
    all_lemmas = lemmatize_text(Sentences)

    h  = '<h3>Scanning for lemmas:</h3>'
    h += '  '.join('[' + lemma + ']' for lemma in all_lemmas)+'<br/>'  
    
    h += '<h3>Datasets descriptions in descending order of score - top '+str(howmany)+':</h3>'
    display(HTML(h))
    
    scores=scan_datasets(all_lemmas,option) 
    idx = np.argsort(scores)[::-1][:howmany] ## indices of top 'howmany' scores
    res2 = crumbs_df.loc[crumbs_df.index[idx],['File description','File code']] ## corresponding records from crumbs_df
    res2['Score'] = scores[idx]
    res2=res2[['Score','File description','File code']]
    res2.reset_index(inplace=True)
    res2.rename(columns={'index':'Dataset id'},inplace=True)

    for i in range(len(res2)):
        url='https://ec.europa.eu/eurostat/estat-navtree-portlet-prod/BulkDownloadListing?sort=1&file=data%2F'+res2.loc[i,'File code']+'.tsv.gz'
        res2.loc[i,'Link'] = url
        
    display(HTML(res2.to_html()))
    
    if show_meta:

        dst = pd.read_csv(res2.loc[show_meta_ind,'Link'],sep='\t',low_memory=False)
        dcode=res2.loc[show_meta_ind,'File code']
        dms = eurostat.get_sdmx_dims(dcode)
        h1 = '<h3>'+str(show_meta_ind)+': File code: '+dcode+'</h3>'
        h1 += '<h3>'+res2.loc[show_meta_ind,'File description']+'</h3>'
        h1 += '<h3>Dimensions:</h3>'
        h1 += '  '.join('[' + x + ']' for x in dms)
        #display(HTML(h1))

        h2=''
        dims=dst.columns[0].split(',')
        dims = dims[:-1]+dims[-1].split('\\')
        #print(dims)
        
        tmp=dst.iloc[:,0].str.split(',',expand=True)
        #print(tmp.head())
        #print('columns:',len(tmp.columns),': ',tmp.columns)
        for j in range(len(dims)-2): ## first the dimensions separated by comma
            if dims[j] not in codes.keys() or codes[dims[j]] != None:
                h2+='<h4>'+dims[j]+':</h4>'
                arr = pd.unique(tmp.iloc[:,j])
                for i in range(len(arr)):
                    h2+= '[ '+arr[i]+': '+eurostat.get_dic(dims[j])[arr[i]] +' ]'
        #j=len(dims)-2 ## omit last dimension which is usually time
        #if codes[dims[j]] is not None:    
        #    h2+='<h4>'+dims[j]+':</h4>'
        #    arr = pd.unique(tmp.iloc[:,j])
        #    for i in range(len(arr)):
        #        h2+= '[ '+arr[i]+': '+eurostat.get_dic(dims[j])[arr[i]] +' ]'
                
                #h2+= '</h4>'
        display(HTML(h1))
        display(HTML(h2))            
        display(HTML(dst.head().to_html()))


## Run once the chunk below then just change inputs

In [ ]:
def relevant_datasets_to_text(): 
    
    first_text = 'In 2021 unemployment among young people in Greece increased. This was the result of recession over the last 10 years.'  
    style = {'description_width': 'initial'}
    
    textW = widgets.Textarea(
        value=first_text, 
        placeholder='Type something',
        description='',
        disabled=False,
        layout=Layout(width='90%', height='100px')
    )

    button = widgets.Button(description="Search")

    option = widgets.RadioButtons(
    options=['Simple description of datasets', 'Full-path description of datasets'],
    value='Full-path description of datasets', 
    layout={'width': 'max-content'}, 
    disabled=False
    )
    
    howmany = widgets.IntSlider(
        description='Display:',
        #tooltip='maximum:',
        value=20,
        min=1, 
        max = 30,
        style=style )
    howmany.style.handle_color = 'lightblue'
    
    show_more = widgets.Checkbox(
        value=True,
        description='Show metadata for row:',
        disabled=False,
        indent=True
     )
    
    select_ind = widgets.BoundedIntText(
        value=0,
        min=0,
        max=howmany.value-1,
        step=1,
        description='Row:',
        disabled=False,
        style={'description_width': 'initial'},
        layout = widgets.Layout(width='100px')
    )

    

    ui1=widgets.HBox([textW])
    ui2=widgets.HBox([option,button,howmany])
    ui3=widgets.HBox([show_more,select_ind])
    ui4=widgets.VBox([ui1,ui2,ui3])
    display(ui4, layout=Layout(align_items='center'))

    def on_button_clicked(b):
        clear_output()
        display(ui4, layout=Layout(align_items='center'))
        if option.value == 'Simple description of datasets':
            opt_val = 1
        else:
            opt_val = 2
        relevant_datasets(textW.value,option=opt_val,howmany=howmany.value,show_meta=show_more.value,show_meta_ind=select_ind.value)
       
    button.on_click(on_button_clicked)
    
    
relevant_datasets_to_text()
   